# Musanga V3.1 · NB05 — Planet NICFI full-year export pipeline

**Overnight, restartable production notebook.** Exports every **available monthly** Planet NICFI B/G/R/N mosaic for **scope-specific reference years** in **two distinct scopes**: (A) the UAV orthomosaic-intersected `ROI.shp` plus 250 m buffer and (B) **one shared 2024 StudyArea** from `ROI_Full.shp`, without cropping the polygon to UAV mosaics. Per scope/site-year, it also exports annual **unscreened and conservatively screened mean + median** reflectance, raw/screened observation counts, screened-out counts, and the **annual median and temporal SD of monthly normalized NIR textures**. Full-resolution data are intended for later spatial CNN experiments.

**Reference years:** UAV_ROI = Baego 2023, MOS 2024, Yangambi 2024, Yoko 2021. FULL_ROI = **2024 for every site**, regardless of UAV acquisition date. This intentionally changes Yoko's Planet imagery from the 2020 UAV acquisition year to 2021, so the model must treat temporal mismatch as a limitation. NICFI monthly imagery begins September 2020, but all selected 2021–2024 years should have twelve months subject to catalogue availability. The bright-observation cloud heuristic is exploratory and **not** an official cloud mask. The Baego pilot removed zero months; do not interpret 'screened' as confirmed cloud-free. Both screened and raw composites are retained. Do not substitute imagery from other years.

**Safety and restart:** At 4.77 m the full StudyArea is exceptionally large (~25 GB uncompressed per 4-band month), and its exports require deliberate `EXPORT_FULL_ROI_HUGE=True`; site UAV exports remain enabled independently. All EE jobs use deterministic names; the runner inspects existing Drive outputs, persisted task IDs and current Earth Engine tasks. Existing Baego Q1/Q2/Q3/Q4 rasters and previous annual/texture files are **preserved** and reused when scope, year and bands match. No delete operation is performed. Exports are submitted in one run without waiting for completion; failed exports can be resubmitted later.

**Run:** 00 → 01 → 02 → 03 → 04 (**dry run**) → set `SUBMIT_TASKS=True` in 00 → rerun 00 and 04. Section 05 monitors tasks and outputs. Set `MAX_NEW_TASKS=None` to submit all pending exports at once (many tasks; quotas still apply).


## 00 · Settings and export policy


In [ ]:
EE_PROJECT='musanga-511110'
SOURCE_COLLECTION='projects/planet-nicfi/assets/basemaps/africa'
UAV_YEAR_BY_SITE={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2021}
FULL_ROI_YEAR=2024
YEAR_BY_SCOPE_SITE={(scope,site):(UAV_YEAR_BY_SITE[site] if scope=='UAV_ROI' else FULL_ROI_YEAR)
                    for scope in ['UAV_ROI','FULL_ROI'] for site in UAV_YEAR_BY_SITE}
SITES=['Baego','MOS','Yangambi','Yoko']
SCOPES=['UAV_ROI','FULL_ROI']
UAV_BUFFER_M=250
FULL_BUFFER_M=0            # ROI_Full boundaries are respected; no UAV footprint clipping
SCALE_M=4.77
EXPORT_CRS='EPSG:32635'
BANDS=['B','G','R','N']
FILE_DIMENSIONS=4096
SUBMIT_TASKS=False        # First inspect Section 04 dry run, then change to True
MAX_NEW_TASKS=None       # None: submit all pending jobs in one go
RESUBMIT_FAILED=True
EXPORT_FULL_ROI_HUGE=False # Explicit opt-in: ROI_Full is ~7.07 million ha (>25 GB per monthly 4-band raster)
FULL_SITE='StudyArea'      # A single shared full-extent polygon, NOT four site copies
MIN_MONTHS_FOR_ANNUAL=2
PRINT_TASK_EVERY=10
print('EE:',EE_PROJECT,'submitting:',SUBMIT_TASKS,'cap:',MAX_NEW_TASKS)
print('Year by scope/site:',YEAR_BY_SCOPE_SITE)


## 01 · Colab, Drive, Earth Engine and persistent output directories


In [ ]:
from google.colab import drive
from pathlib import Path
import ee,geopandas as gpd,rasterio,pandas as pd,numpy as np
import json,datetime,hashlib,os,math,time
from shapely.geometry import box,mapping
from shapely.ops import unary_union
from IPython.display import display

if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
FINAL=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert FINAL, 'Musanga/Final missing'
DATA=FINAL/'Data/Ortho_RGB'
OUT=FINAL/'Outputs/V3_1/05_sat_predictor_exploration'
IN=OUT/'input_layers'
DER=OUT/'derived_features'
META=OUT/'metadata'
for p in [IN,DER,META]:p.mkdir(parents=True,exist_ok=True)
# The EE toDrive folder argument is a NAME, not a folder path. Every folder name below is unique.
FOLDERS={}
for scope in SCOPES:
    for family in ('Monthly','Annual','Texture'):
        dirname=f'Musanga_V31_NB05_{scope}_{family}'
        parent=DER if family=='Texture' else IN
        dest=parent/dirname
        dest.mkdir(parents=True,exist_ok=True)
        FOLDERS[(scope,family)]=(dirname,dest)
# Existing pilot cache (preserve; do not copy into a different output folder).
LEGACY_MONTHLY=IN/'Musanga_V3_1_05_Planet_NICFI_Monthly'
LEGACY_ANNUAL=IN/'Musanga_V3_1_05_Planet_Annual'
LEGACY_TEXTURE=OUT/'derived_features/Musanga_V3_1_05_Planet_Texture'
LEGACY_REGISTRY=OUT/'05_export_registry_nested_input_layers_v2.json'
assert EE_PROJECT
ee.Authenticate(auth_mode='colab')
ee.Initialize(project=EE_PROJECT)
NICFI=ee.ImageCollection(SOURCE_COLLECTION)
print('Workspace:',FINAL)
print('Earth Engine access:',NICFI.limit(1).size().getInfo(),'image test')
print('Export folders (must be uniquely named in Drive):')
for k,(name,folder) in FOLDERS.items(): print(k,folder)


## 02 · Audit the **two** ROI geometries and recognize one shared full-ROI region

UAV scope: `ROI.shp` intersected with each registered RGB orthomosaic **bounding footprint**, buffered 250 m.

Full scope: `Final/Data/Ortho_RGB/ROI_Full.shp` is **one shared region** spanning all four sites; export it once as `FULL_ROI/StudyArea/2024`. It is not split, assigned to a site or duplicated.


In [ ]:
MANIFEST=FINAL/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert MANIFEST.is_file(),f'Missing NB00 manifest: {MANIFEST}'
manifest=json.loads(MANIFEST.read_text())
roi_path=Path(manifest['roi'])
full_path=DATA/'ROI_Full.shp'
assert roi_path.is_file(),roi_path
assert full_path.is_file(),f'Full ROI shapefile missing: {full_path}'
roi=gpd.read_file(roi_path)
full=gpd.read_file(full_path)
assert roi.crs is not None and full.crs is not None
roi=roi.loc[roi.geometry.notna() & ~roi.geometry.is_empty].copy()
full=full.loc[full.geometry.notna() & ~full.geometry.is_empty].copy()
assert len(roi) and len(full)
NAMES={'Baego.tif':'Baego',
       'Mos_1_2_transparent_mosaic_group1_georef.tif':'MOS',
       'Mos_5_6_transparent_mosaic_group1_georef.tif':'MOS',
       'Yangambi_georef.tif':'Yangambi',
       'Yoko_kvo_georef.tif':'Yoko'}
mosaic_parts={s:[] for s in SITES}
uav_parts={s:[] for s in SITES}
record=[]
for item in manifest['mosaics']:
    name=item['file']
    if name not in NAMES:continue
    site=NAMES[name]
    with rasterio.open(item['path']) as ds:
        assert ds.crs,name
        footprint=gpd.GeoSeries([box(*ds.bounds)],crs=ds.crs).to_crs(EXPORT_CRS).iloc[0]
        local_roi=roi.to_crs(ds.crs)
        intersection=unary_union(local_roi.geometry).intersection(box(*ds.bounds))
        mosaic_parts[site].append(footprint)
        if not intersection.is_empty:
            uav_parts[site].append(gpd.GeoSeries([intersection],crs=ds.crs).to_crs(EXPORT_CRS).iloc[0])
        record.append(dict(site=site,mosaic=name,uav_intersection_ha=round(intersection.area/1e4,2)))
display(pd.DataFrame(record))
site_fp={s:unary_union(mosaic_parts[s]) for s in SITES}
AOI_METRIC={}
for s in SITES:
    assert uav_parts[s],f'No UAV ROIs at {s}'
    AOI_METRIC[('UAV_ROI',s)]=unary_union(uav_parts[s]).buffer(UAV_BUFFER_M)
# ROI_Full.shp is ONE common polygon spanning all forests, not site-wise features.
# Preserve the entire geometry exactly once; never assign it to individual sites.
full_metric=full.to_crs(EXPORT_CRS)
full_geom=unary_union(full_metric.geometry)
assert full_geom.is_valid and not full_geom.is_empty
AOI_METRIC[('FULL_ROI',FULL_SITE)]=full_geom.buffer(FULL_BUFFER_M) if FULL_BUFFER_M else full_geom
audit_full=pd.DataFrame([dict(scope='FULL_ROI',site=FULL_SITE,features=len(full_metric),
    polygon_ha=round(full_geom.area/1e4,2),note='one shared 2024 extent (no site assignment)')])
display(audit_full)
audit_full.to_csv(META/'02_roi_full_assignment.csv',index=False)

AOI={}
area_rows=[]
for (scope,site),geom in AOI_METRIC.items():
    assert geom.is_valid and not geom.is_empty,(scope,site)
    geographic=gpd.GeoSeries([geom.simplify(1,preserve_topology=True)],crs=EXPORT_CRS).to_crs('EPSG:4326').iloc[0]
    AOI[(scope,site)]=ee.Geometry(mapping(geographic),geodesic=False)
    area_rows.append(dict(scope=scope,site=site,area_ha=round(geom.area/1e4,2),
        approx_4band_uncompressed_GB_per_month=round(geom.area/(SCALE_M*SCALE_M)*8/1e9,3)))
area_df=pd.DataFrame(area_rows)
display(area_df)
area_df.to_csv(META/'02_roi_scope_areas.csv',index=False)
(META/'02_roi_scope_geometries.geojson').write_text(json.dumps({
    'type':'FeatureCollection','features':[dict(type='Feature',
        properties=dict(scope=k[0],site=k[1]),geometry=mapping(
            gpd.GeoSeries([v],crs=EXPORT_CRS).to_crs('EPSG:4326').iloc[0]))
        for k,v in AOI_METRIC.items()]}))
full_gb=float(full_geom.area/(SCALE_M*SCALE_M)*8/1e9)
print(f'FULL ROI at {SCALE_M}m: ~{full_gb:,.1f} GB UNCOMPRESSED per 4-band month, ~{12*full_gb:,.1f} GB for 12 monthly products.')
print('Export full-area products once only; no duplication across forests.')
print('FULL ROI huge export enabled:', EXPORT_FULL_ROI_HUGE)
print('ROI_Full used at full polygon extent; no clipping to the UAV mosaic.')


## 03 · Monthly NICFI catalogue (five independent scope/year combinations)

UAV_ROI: four site-specific years (Baego 2023, MOS/Yangambi 2024, Yoko 2021). FULL_ROI: **one shared StudyArea in 2024**. Do not duplicate the full ROI for each site. Missing months remain missing. Full ROI catalogue uses one representative bounds-based collection query over the true full footprint.


In [ ]:
def month_dates(year,month):
    begin=datetime.date(year,month,1)
    end=datetime.date(year+(month==12),month%12+1,1)
    return begin.isoformat(),end.isoformat()
MONTHS={}
rows=[]
COMBINATIONS=[('UAV_ROI',site) for site in SITES]+[('FULL_ROI',FULL_SITE)]
for scope,site in COMBINATIONS:
    year=UAV_YEAR_BY_SITE[site] if scope=='UAV_ROI' else FULL_ROI_YEAR
    for month in range(1,13):
        start,end=month_dates(year,month)
        subset=(NICFI.filterDate(start,end)
                .filter(ee.Filter.eq('cadence','monthly'))
                .filterBounds(AOI[(scope,site)]))
        count=int(subset.size().getInfo())
        row=dict(scope=scope,site=site,year=year,month=month,n_images=count)
        if count==1:
            image=ee.Image(subset.first())
            info=ee.Dictionary(dict(id=image.id(),bands=image.bandNames())).getInfo()
            assert set(BANDS).issubset(set(info['bands'])),(scope,site,month)
            MONTHS[(scope,site,month)]=image.select(BANDS)
            row.update(status='available',image_id=info['id'])
        else:
            row.update(status='missing' if count==0 else 'AMBIGUOUS',image_id=None)
        rows.append(row)
catalogue=pd.DataFrame(rows)
display(catalogue.pivot_table(index='month',columns=['scope','site'],values='status',aggfunc='first'))
catalogue.to_csv(META/'03_monthly_catalogue.csv',index=False)
assert not (catalogue.status=='AMBIGUOUS').any(),'Monthly catalogue ambiguous: inspect'
for scope,site in COMBINATIONS:
    available=[m for m in range(1,13) if (scope,site,m) in MONTHS]
    yr=UAV_YEAR_BY_SITE[site] if scope=='UAV_ROI' else FULL_ROI_YEAR
    print(scope,site,yr,f'{len(available)} months:',available)
    assert len(available)>=MIN_MONTHS_FOR_ANNUAL


## 04 · **One-click batch submission** — monthly rasters + annual mean/median + NIR texture

One cell submits **all pending** jobs for four UAV sites plus one shared full-ROI study area, with reference years set per scope. Save registry to Drive **immediately after each submitted job**. Checks actual cached files, old Baego pilot files, registry and live EE task descriptions to avoid duplicates. Skips complete exports, but a completed task with no visible files is flagged for inspection rather than blindly duplicated. EE may queue many jobs and enforce user/project quota; re-run to pick up failed jobs.

**Filename convention**: `NB05_{scope}_{site}_{year}_M{MM}_DN4.tif`, `NB05_{scope}_{site}_{year}_annual_...tif`. A legacy Baego quarterly file can satisfy a UAV-scope monthly job when its provenance matches. The historical Baego annual/texture are reused for UAV_ROI only (they have different band definitions, and are registered as legacy rather than mislabelled as the new products).


In [ ]:
# 04 — Definitions and restartable task registry. Run with SUBMIT_TASKS=False to audit.
REGISTRY_FILE=META/'04_overnight_export_registry.json'
registry=json.loads(REGISTRY_FILE.read_text()) if REGISTRY_FILE.is_file() else {}
def save_registry():
    tmp=REGISTRY_FILE.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(registry,indent=2,sort_keys=True))
    os.replace(tmp,REGISTRY_FILE)
def states():
    out={}
    try:
        for task in ee.batch.Task.list():
            stat=task.status()
            if stat.get('description'):out[stat['description']]=dict(id=task.id,state=stat.get('state','UNKNOWN'))
    except Exception as exc:print('EE task list unavailable:',repr(exc))
    return out
LIVE=states()
def exists_output(scope,site,year,kind,month=None):
    foldername,dest=FOLDERS[(scope,kind)]
    if kind=='Monthly':
        prefix=f'NB05_{scope}_{site}_{year}_M{month:02d}_DN4'
    elif kind=='Annual':
        prefix=f'NB05_{scope}_{site}_{year}_Annual_19band'
    else:
        prefix=f'NB05_{scope}_{site}_{year}_Texture_9band'
    paths=sorted(dest.glob(prefix+'*.tif'))
    legacy=[]
    if scope=='UAV_ROI' and site=='Baego' and year==2023 and kind=='Monthly':
        if month in (3,6,9,12):
            q={3:1,6:2,9:3,12:4}[month]
            legacy=sorted(LEGACY_MONTHLY.glob(f'NB05_Planet_Baego_2023_Q{q}_M{month:02d}_DN4*.tif'))
        else:
            legacy=sorted(LEGACY_MONTHLY.glob(f'NB05_Planet_Baego_2023_M{month:02d}_DN4*.tif'))
    # Historical annual/texture products have different band layouts and must NOT
    # masquerade as the new 19-band annual / 9-band scoped products.
    return prefix,paths,legacy

def month_collection(scope,site,region):
    ims=[MONTHS[(scope,site,m)] for m in range(1,13) if (scope,site,m) in MONTHS]
    return ee.ImageCollection.fromImages(ims).map(
        lambda im: im.updateMask(im.reduce(ee.Reducer.min()).gt(0)).clip(region))
def process_annual(site,scope):
    region=AOI[(scope,site)]
    raw=month_collection(scope,site,region)
    reference=raw.median()
    # Haze/cloud heuristic, not UDM2. Keep screened + unscreened annual products.
    def screen(im):
        vis=im.select(['B','G','R']).reduce(ee.Reducer.mean()).multiply(0.0001)
        base=reference.select(['B','G','R']).reduce(ee.Reducer.mean()).multiply(0.0001)
        ndvi=im.normalizedDifference(['N','R'])
        bndvi=reference.normalizedDifference(['N','R'])
        bad=vis.subtract(base).gt(0.08).And(bndvi.subtract(ndvi).gt(0.10))
        return im.updateMask(bad.Not())
    screened=raw.map(screen)
    # 16 bands: 4 screened mean, 4 screened median, 4 raw mean,
    # 4 raw median, followed by 3 observation counts = 19 bands.
    means=screened.mean().rename([x+'_screen_mean' for x in BANDS])
    medians=screened.median().rename([x+'_screen_median' for x in BANDS])
    raw_mean=raw.mean().rename([x+'_raw_mean' for x in BANDS])
    raw_median=raw.median().rename([x+'_raw_median' for x in BANDS])
    nr=raw.select('N').count().rename('months_raw_valid')
    ns=screened.select('N').count().rename('months_screened_valid')
    nm=nr.subtract(ns).rename('months_screened_out')
    annual=ee.Image.cat([means,medians,raw_mean,raw_median,nr,ns,nm]).toFloat().clip(region)
    def texture(im):
        nir=im.select('N').multiply(0.0001)
        layers=[]
        for radius in (1,3,7):
            kernel=ee.Kernel.square(radius=radius,units='pixels',normalize=False)
            avg=nir.reduceNeighborhood(ee.Reducer.mean(),kernel)
            sd=nir.reduceNeighborhood(ee.Reducer.stdDev(),kernel)
            layers.append(sd.divide(avg.abs().max(ee.Image.constant(0.02))).rename(f'NIR_CV_r{radius}'))
        mean5=nir.focal_mean(radius=2,units='pixels')
        high=nir.subtract(mean5).abs().divide(mean5.abs().max(ee.Image.constant(0.02)))
        return ee.Image.cat(layers+[high.rename('NIR_abs_highpass_r2')]).toFloat()
    tcol=screened.map(texture)
    tm=tcol.median()
    names=['NIR_CV_r1','NIR_CV_r3','NIR_CV_r7','NIR_abs_highpass_r2']
    ts=tcol.reduce(ee.Reducer.stdDev()).rename([x+'_temporal_sd' for x in names])
    nt=tcol.select('NIR_CV_r3').count().rename('texture_valid_month_count')
    tex=ee.Image.cat([tm,ts,nt]).toFloat().clip(region)
    return annual,tex

def export_one(key,scope,site,year,kind,image,month=None):
    global n_submitted
    prefix,parts,legacy=exists_output(scope,site,year,kind,month)
    desc=f'MUSANGA_V31_NB05_{scope}_{site}_{year}_'+(f'M{month:02d}' if month else kind.upper())
    rec=registry.get(key,{})
    previous=LIVE.get(desc,{})
    if parts or legacy:
        if key not in registry:
            registry[key]=dict(status='cached',paths=[str(p) for p in (parts or legacy)],
                               prefix=prefix,description=desc)
            save_registry()
        print('CACHED',key,'pieces',len(parts or legacy))
        return
    state=previous.get('state')
    if not state and rec.get('task_id'):
        try:
            st=ee.data.getTaskStatus(rec['task_id'])
            state=st[0].get('state','UNKNOWN') if st else None
        except Exception:state=None
    if state in ('READY','RUNNING','COMPLETED'):
        print('TASK',key,state,'(completed but not visible: inspect Drive folder)' if state=='COMPLETED' else '')
        return
    if rec.get('task_id') and state is None and not RESUBMIT_FAILED:
        print('UNKNOWN prior job:',key);return
    if not SUBMIT_TASKS:
        print('PENDING',key,'prior state',state or 'none');return
    if MAX_NEW_TASKS is not None and n_submitted>=MAX_NEW_TASKS:
        print('DEFERRED cap',key);return
    foldername,folder=FOLDERS[(scope,kind)]
    task=ee.batch.Export.image.toDrive(
        image=image,description=desc,folder=foldername,fileNamePrefix=prefix,
        region=AOI[(scope,site)],crs=EXPORT_CRS,scale=SCALE_M,
        fileDimensions=FILE_DIMENSIONS,shardSize=256,skipEmptyTiles=True,
        maxPixels=1e13,fileFormat='GeoTIFF',formatOptions={'cloudOptimized':True})
    task.start()
    registry[key]=dict(status='submitted',task_id=task.id,description=desc,
        scope=scope,site=site,year=year,kind=kind,month=month,
        folder=foldername,prefix=prefix,crs=EXPORT_CRS,scale_m=SCALE_M,
        band_count=4 if kind=='Monthly' else (19 if kind=='Annual' else 9))
    save_registry()
    LIVE[desc]=dict(id=task.id,state='READY')
    n_submitted+=1
    print('SUBMITTED',n_submitted,key,task.id,flush=True)
    if PRINT_TASK_EVERY and n_submitted%PRINT_TASK_EVERY==0:
        print('Checkpoint registry:',REGISTRY_FILE,flush=True)

n_submitted=0
planned=0
for scope,site in COMBINATIONS:
    year=UAV_YEAR_BY_SITE[site] if scope=='UAV_ROI' else FULL_ROI_YEAR
    if scope=='FULL_ROI' and not EXPORT_FULL_ROI_HUGE:
        print('SKIP HUGE full ROI: enable EXPORT_FULL_ROI_HUGE=True only after reviewing ~25 GB/month footprint.')
        continue
    for month in range(1,13):
        if (scope,site,month) not in MONTHS:continue
        planned+=1
        key=f'{scope}/{site}/{year}/monthly/{month:02d}'
        image=MONTHS[(scope,site,month)].toUint16().clip(AOI[(scope,site)])
        export_one(key,scope,site,year,'Monthly',image,month)
    annual,texture=process_annual(site,scope)
    planned+=2
    export_one(f'{scope}/{site}/{year}/annual',scope,site,year,'Annual',annual)
    export_one(f'{scope}/{site}/{year}/texture',scope,site,year,'Texture',texture)
print('Planned available-month products:',planned,'| New tasks:',n_submitted)
print('Full ROI export enabled:',EXPORT_FULL_ROI_HUGE,'registry:',REGISTRY_FILE)


## 05 · Restart, check progress and verify actual files

Rerun 00–04 whenever Colab restarts. The pipeline uses the saved registry and EE live tasks to avoid duplicate submissions. **Completed EE task alone is not proof that a GeoTIFF exists in the expected Drive folder.** Check output sizes and GeoTIFF bands. Large full-ROI GeoTIFFs may consist of multiple parts. This QA checks existence and band metadata, not pixel-level science.


In [ ]:
# 05 — Task and Drive audit, safe to rerun.
if not REGISTRY_FILE.is_file():
    print('No exports yet.')
else:
    reg=json.loads(REGISTRY_FILE.read_text())
    audit=[]
    for key,item in sorted(reg.items()):
        if item.get('status')=='cached':
            paths=[Path(x) for x in item.get('paths',[])]
            state='CACHED'
        else:
            folder=FOLDERS[(item['scope'],item['kind'])][1]
            paths=sorted(folder.glob(item['prefix']+'*.tif'))
            try:
                st=ee.data.getTaskStatus(item['task_id'])
                state=st[0].get('state','UNKNOWN') if st else 'UNKNOWN'
            except Exception:
                state='UNKNOWN'
        audit.append(dict(job=key,ee_state=state,files=len([x for x in paths if x.exists()]),
            size_MB=round(sum(x.stat().st_size for x in paths if x.exists())/1e6,2)))
    a=pd.DataFrame(audit)
    display(a.groupby('ee_state',dropna=False).agg(jobs=('job','size'),files=('files','sum'),
                                                   total_MB=('size_MB','sum')).reset_index())
    display(a.loc[(a.files==0)|(a.ee_state=='FAILED')].head(100))
    a.to_csv(META/'05_drive_export_audit.csv',index=False)
    print('CSV:',META/'05_drive_export_audit.csv')


## 06 · Scientific next steps (after exports)

1. Inspect annual screened mean vs median vs unscreened median, and observation-count surfaces. Since the Baego pilot's heuristic mask excluded no months, call it **conservative screening**, not validated cloud removal.
2. Compare annual median monthly-NIR texture with monthly-image consistency, and measure cloud/registration sensitivity before training CNNs.
3. Train native 4.77 m Planet patch CNNs against UAV-derived 30/50/100 m fractions, carefully aligning CRS and grid; include site holdout when moving from feasibility to inference.
4. The FULL_ROI exports are spatial-context products for later inference; their large footprints are **not** UAV-labelled training data.


## 07 · FINAL handoff audit: verify Planet data before NB06

**Read-only** (apart from saved audit tables/figures). Does **not** submit Earth Engine tasks or download imagery. Checks all four UAV forest sites for 12 monthly images, correct observation years, 4/19/9-band counts, coordinate systems, raster footprints, positive valid pixels and annual observation-count consistency. Generates visual comparison figures for annual RGB and annual NIR texture, using the same radiometric scale per site and masking exported background. Distinguishes *legacy Baego cached* monthly files from the standardized output folder and checks their geometry as part of the site audit.

**Footprint coverage** is now calculated blockwise using every raster pixel whose center lies within the **unbuffered ROI.shp ∩ original UAV orthomosaic footprints**. This avoids counting white background or the 250 m Planet export buffer as missing data. It checks all four monthly bands, annual valid-month counts and annual texture valid-month counts. Run Sections 00–02 before Section 07. The quicklook still uses reduced-resolution imagery. Finds candidate UAV probability rasters, but **does not claim that filenames alone establish correct supervision**; NB06 must inspect alignment with the original UAV model output.

**Success criterion:** every forest has 12 valid monthly rasters and both annual products, with sensible geometry/coverage. Save `07_input_audit.csv`, `07_site_summary.csv`, `07_monthly_file_manifest.csv`, and `07_annual_quicklooks.png`.


In [ ]:
# 07 — Read-only final audit. Can run after Section 01; does not need EE/ROI cells.
from pathlib import Path
import rasterio, numpy as np, pandas as pd, matplotlib.pyplot as plt
from rasterio.enums import Resampling
from rasterio.windows import Window
from rasterio.features import geometry_mask
from rasterio.windows import transform as window_transform
from shapely.ops import unary_union
from IPython.display import display

AUDIT_DIR=OUT/'07_final_input_audit'
AUDIT_DIR.mkdir(parents=True,exist_ok=True)
OBS_YEAR={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2021}
EXPECTED={'Monthly':4,'Annual':19,'Texture':9}
audit_rows=[]
site_summaries=[]
month_manifest=[]
annual_sources={}

# True unbuffered UAV reference footprint: ROI.shp intersected with the UAV
# mosaic bounds (Section 02), NOT the buffered Planet export geometry.
# Section 02 must have run in this Colab session.
assert 'uav_parts' in globals() and all(uav_parts.get(site) for site in OBS_YEAR), (
    'Run Section 02 before Section 07: original UAV ROI/orthomosaic intersections required.'
)
UAV_REFERENCE_FOOTPRINTS = {site:unary_union(uav_parts[site]) for site in OBS_YEAR}

def footprint_coverage(path,site,kind):
    """Exact raster-pixel-center coverage inside unbuffered UAV ROI, blockwise.

    Counts all area inside the true UAV footprint, including source NoData pixels.
    Valid: Monthly B/G/R/N > 0; Annual raw month count > 0 (band 18);
    Texture monthly valid count > 0 (band 9). No padding is counted.
    """
    geom=UAV_REFERENCE_FOOTPRINTS[site]
    total=valid=0
    count_total=0.
    count_atleast_9=0
    if geom.is_empty:
        return dict(footprint_coverage_pct=np.nan,footprint_pixels=0,
                    footprint_valid_pixels=0,footprint_mean_valid_months=np.nan,
                    footprint_pct_atleast_9_months=np.nan)
    with rasterio.open(path) as ds:
        if not ds.crs:
            raise ValueError(f'Missing CRS: {path}')
        # Section 02's footprint geometries are in EXPORT_CRS (EPSG:32635).
        if ds.crs.to_string()!=EXPORT_CRS:
            footprint=gpd.GeoSeries([geom],crs=EXPORT_CRS).to_crs(ds.crs).iloc[0]
        else:
            footprint=geom
        shapes=[footprint.__geo_interface__]
        for _,window in ds.block_windows(1):
            wt=window_transform(window,ds.transform)
            within=geometry_mask(shapes,out_shape=(int(window.height),int(window.width)),
                                 transform=wt,invert=True,all_touched=False)
            n=int(within.sum())
            if n==0:continue
            total+=n
            if kind=='Monthly':
                a=ds.read([1,2,3,4],window=window)
                ok=np.isfinite(a).all(axis=0) & (a>0).all(axis=0)
            elif kind=='Annual':
                counts=ds.read(18,window=window)
                ok=np.isfinite(counts)&(counts>0)
                count_total+=float(counts[within & np.isfinite(counts)].sum())
                count_atleast_9+=int((within & (counts>=9)).sum())
            elif kind=='Texture':
                counts=ds.read(9,window=window)
                ok=np.isfinite(counts)&(counts>0)
                count_total+=float(counts[within & np.isfinite(counts)].sum())
                count_atleast_9+=int((within & (counts>=9)).sum())
            else:
                raise ValueError(kind)
            valid+=int((within&ok).sum())
    return dict(footprint_coverage_pct=100*valid/total if total else np.nan,
                footprint_pixels=total,footprint_valid_pixels=valid,
                footprint_mean_valid_months=(count_total/total if total and kind!='Monthly' else np.nan),
                footprint_pct_atleast_9_months=(100*count_atleast_9/total if total and kind!='Monthly' else np.nan))

def audit_file(filepath,expected_bands):
    """Inspect metadata and representative valid pixels without full raster read."""
    if not filepath or not Path(filepath).is_file():
        return dict(status='MISSING',path=str(filepath) if filepath else '',bands=0,
                    crs='',width=0,height=0,valid_sample_pct=np.nan,bytes=0)
    path=Path(filepath)
    try:
        with rasterio.open(path) as ds:
            bands=ds.count
            crs=str(ds.crs)
            n_samples=0; n_valid=0
            for fy,fx in ((.15,.15),(.15,.50),(.15,.85),(.50,.15),(.50,.50),
                          (.50,.85),(.85,.15),(.85,.50),(.85,.85)):
                yy=min(max(0,int(ds.height*fy)),ds.height-1)
                xx=min(max(0,int(ds.width*fx)),ds.width-1)
                win=Window(max(0,xx-32),max(0,yy-32),
                           min(64,ds.width-max(0,xx-32)),
                           min(64,ds.height-max(0,yy-32)))
                a=ds.read(1,window=win,masked=False)
                n_valid+=int(np.count_nonzero(np.isfinite(a)&(a!=0)))
                n_samples+=a.size
            status='OK' if bands==expected_bands and ds.crs and ds.width>0 and ds.height>0 else 'CHECK_METADATA'
            return dict(status=status,path=str(path),bands=bands,crs=crs,
                        width=ds.width,height=ds.height,transform=str(ds.transform),
                        bounds=str(tuple(round(x,1) for x in ds.bounds)),
                        pixel_size_x=abs(ds.res[0]),pixel_size_y=abs(ds.res[1]),
                        valid_sample_pct=100*n_valid/max(n_samples,1),
                        bytes=path.stat().st_size,nodata=str(ds.nodata))
    except Exception as exc:
        return dict(status='UNREADABLE',path=str(path),error=repr(exc),
                    bands=0,crs='',width=0,height=0,valid_sample_pct=np.nan,bytes=path.stat().st_size)

def discover(scope,site,year,kind,month=None):
    folder=FOLDERS[(scope,kind)][1]
    if kind=='Monthly':
        prefix=f'NB05_{scope}_{site}_{year}_M{month:02d}_DN4'
    elif kind=='Annual':
        prefix=f'NB05_{scope}_{site}_{year}_Annual_19band'
    else:
        prefix=f'NB05_{scope}_{site}_{year}_Texture_9band'
    matched=sorted(folder.glob(prefix+'*.tif'))
    if not matched and kind=='Monthly' and scope=='UAV_ROI' and site=='Baego' and year==2023:
        if month in (3,6,9,12):
            q={3:1,6:2,9:3,12:4}[month]
            prefix=f'NB05_Planet_Baego_2023_Q{q}_M{month:02d}_DN4'
        else:
            prefix=f'NB05_Planet_Baego_2023_M{month:02d}_DN4'
        matched=sorted(LEGACY_MONTHLY.glob(prefix+'*.tif'))
    return matched

for site,year in OBS_YEAR.items():
    monthly_ok=0
    valid_sizes=[]
    monthly_meta=[]
    for month in range(1,13):
        files=discover('UAV_ROI',site,year,'Monthly',month)
        if len(files)==1:
            record=audit_file(files[0],4)
        elif len(files)==0:
            record=dict(status='MISSING',path='',bands=0,crs='',width=0,height=0,
                        valid_sample_pct=np.nan,bytes=0)
        else:
            record=dict(status='MULTIPART_CHECK',path=';'.join(map(str,files)),
                        bands=0,crs='',width=0,height=0,valid_sample_pct=np.nan,
                        bytes=sum(p.stat().st_size for p in files))
        if len(files)==1 and record['status']=='OK':
            record.update(footprint_coverage(files[0],site,'Monthly'))
        audit_rows.append(dict(site=site,year=year,kind='Monthly',month=month,parts=len(files),**record))
        month_manifest.append(dict(site=site,year=year,month=month,files=';'.join(map(str,files)),
                                   status=record['status']))
        if record['status']=='OK':
            monthly_ok+=1
            monthly_meta.append(record)
            valid_sizes.append(record.get('footprint_coverage_pct',np.nan))
    for kind in ('Annual','Texture'):
        files=discover('UAV_ROI',site,year,kind)
        if len(files)==1:
            rec=audit_file(files[0],EXPECTED[kind])
            annual_sources[(site,kind)]=files[0]
        elif not files:
            rec=dict(status='MISSING',path='',bands=0,crs='',width=0,height=0,
                     valid_sample_pct=np.nan,bytes=0)
        else:
            rec=dict(status='MULTIPART_CHECK',path=';'.join(map(str,files)),
                     bands=0,crs='',width=0,height=0,valid_sample_pct=np.nan,
                     bytes=sum(p.stat().st_size for p in files))
        if len(files)==1 and rec['status']=='OK':
            rec.update(footprint_coverage(files[0],site,kind))
        audit_rows.append(dict(site=site,year=year,kind=kind,month=np.nan,parts=len(files),**rec))
    annual_ok=all((site,kind) in annual_sources and
                  next(z for z in audit_rows if z['site']==site and z['kind']==kind)['status']=='OK'
                  for kind in ('Annual','Texture'))
    # Pixel footprints may differ slightly because an earlier Baego pilot used a
    # different EE export grid. Warn, don't forcibly resample source data.
    crs_set=set(x['crs'] for x in monthly_meta)
    width_set=set(x['width'] for x in monthly_meta)
    height_set=set(x['height'] for x in monthly_meta)
    summary=dict(site=site,year=year,monthly_OK=monthly_ok,monthly_expected=12,
                 annual_products_OK=annual_ok,
                 monthly_crs_consistent=(len(crs_set)==1),
                 monthly_dimensions_consistent=(len(width_set)==1 and len(height_set)==1),
                 mean_monthly_footprint_coverage_pct=float(np.nanmean(valid_sizes)) if valid_sizes else np.nan,
                 minimum_monthly_footprint_coverage_pct=float(np.nanmin(valid_sizes)) if valid_sizes else np.nan)
    relevant=[z for z in audit_rows if z['site']==site]
    for kind in ('Annual','Texture'):
        match=next((z for z in relevant if z['kind']==kind),None)
        summary[kind.lower()+'_footprint_coverage_pct']=(match or {}).get('footprint_coverage_pct',np.nan)
        summary[kind.lower()+'_mean_valid_months']=(match or {}).get('footprint_mean_valid_months',np.nan)
    summary['status']='READY_FOR_VISUAL_QA' if monthly_ok==12 and annual_ok else 'INCOMPLETE_OR_CHECK'
    site_summaries.append(summary)

detail=pd.DataFrame(audit_rows)
summary=pd.DataFrame(site_summaries)
detail.to_csv(AUDIT_DIR/'07_input_audit.csv',index=False)
summary.to_csv(AUDIT_DIR/'07_site_summary.csv',index=False)
pd.DataFrame(month_manifest).to_csv(AUDIT_DIR/'07_monthly_file_manifest.csv',index=False)
display(summary)
display(detail.loc[detail.status!='OK', ['site','year','kind','month','parts','status','path']])
print('Total nominal monthly rasters:',int((detail.kind=='Monthly').sum()))
print('Coverage denominator: unbuffered UAV ROI ∩ orthomosaic footprint; valid monthly pixels have all 4 bands > 0.')
print('Monthly coverage by site and month (%)')
display(detail.loc[detail.kind=='Monthly'].pivot(index='month',columns='site',values='footprint_coverage_pct').round(2))
print('Annual raw valid-month count and texture valid-month count measured within true UAV footprint.')
print('Saved QA tables:',AUDIT_DIR)

# Annual products are small enough to render via reduced-resolution sampling.
fig,axes=plt.subplots(len(OBS_YEAR),3,figsize=(13,4*len(OBS_YEAR)),layout='constrained')
for row,(site,year) in enumerate(OBS_YEAR.items()):
    f_annual=annual_sources.get((site,'Annual'))
    f_texture=annual_sources.get((site,'Texture'))
    for ax in axes[row,:]:
        ax.set_xticks([]);ax.set_yticks([])
    if not f_annual or not f_texture:
        for ax in axes[row,:]: ax.text(.5,.5,'MISSING ANNUAL PRODUCT',
                                      ha='center',va='center',transform=ax.transAxes)
        continue
    try:
        with rasterio.open(f_annual) as ds:
            factor=max(1,int(np.ceil(max(ds.width,ds.height)/1100)))
            h=max(1,int(np.ceil(ds.height/factor)))
            w=max(1,int(np.ceil(ds.width/factor)))
            # Annual order: screened mean B/G/R/N (1..4), screened median (5..8),
            # raw mean (9..12), raw median (13..16), 3 count bands.
            a=ds.read([7,6,5],out_shape=(3,h,w),
                      resampling=Resampling.average).astype('float32')
            b=ds.read([15,14,13],out_shape=(3,h,w),
                      resampling=Resampling.average).astype('float32')
            count=ds.read(18,out_shape=(h,w),resampling=Resampling.nearest)
            fullvalid=count>0
            if not fullvalid.any():
                raise ValueError('Annual valid-count band contains no positive observations')
            vals=np.concatenate([a[:,fullvalid],b[:,fullvalid]],axis=1)
            low=np.percentile(vals,2,axis=1)
            high=np.percentile(vals,98,axis=1)
            def rgb(x):
                x=np.moveaxis(np.clip((x-low[:,None,None])/(high[:,None,None]-low[:,None,None]+1e-6),0,1),0,-1)
                x[~fullvalid]=1.
                return x
            axes[row,0].imshow(rgb(b))
            axes[row,0].set_title(f'{site} {year} — raw median')
            axes[row,1].imshow(rgb(a))
            axes[row,1].set_title('Screened median')
        with rasterio.open(f_texture) as ds:
            tx=ds.read(2,out_shape=(h,w),resampling=Resampling.average).astype('float32')
            tx=np.where(fullvalid,tx,np.nan)
            axes[row,2].imshow(np.ma.masked_invalid(tx),cmap='viridis',
                               vmin=float(np.nanpercentile(tx,2)),
                               vmax=float(np.nanpercentile(tx,98)))
            axes[row,2].set_title('Annual NIR CV, radius 3 px')
    except Exception as exc:
        for ax in axes[row,:]:ax.clear();ax.text(.05,.5,str(exc)[:140],transform=ax.transAxes)
fig.savefig(AUDIT_DIR/'07_annual_quicklooks.png',dpi=140,bbox_inches='tight')
plt.show()
print('NOTE: Annual cloud-screening is an unvalidated heuristic.')
print('NB06 should independently audit UAV teacher-target co-registration before training.')
